# SNLI-VE × CLIP — Embedding Extraction & PCA Visualization

**Goal:** Step-by-step walkthrough of the `scripts/extract_snli_ve_clip.py` pipeline inside a notebook.

**Steps covered:**
1. Load the SNLI-VE dataset and inspect its schema
2. Build a balanced 3-class subset (1 000 per class, seed 42)
3. Load `openai/clip-vit-base-patch32` (no fine-tuning needed — CLIP is used as a frozen feature extractor)
4. Extract and L2-normalise image + text embeddings
5. Build the four representations: image, text, concat, pairwise
6. Save to `data/embeddings/snli_ve_clip.npz`
7. PCA 2D scatter plots — one panel per representation

> **Fine-tuning?** CLIP is used *zero-shot* here: we extract frozen features and analyse their geometry.  
> Fine-tuning CLIP on SNLI-VE would require a contrastive or classification objective and is out of scope for this pipeline.

## 0 · Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import random
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from datasets import load_dataset
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

from features_extraction import setup_logging, save_features
from features_extraction.device import DeviceManager

setup_logging(level="WARNING")  # suppress INFO from features_extraction during notebook run

# ── Constants ─────────────────────────────────────────────────────────────────
MODEL_NAME  = "openai/clip-vit-base-patch32"
HF_ID       = "pingzhili/snli-ve"
SPLIT       = "train"
N_PER_CLASS = 1_000
SEED        = 42
BATCH_SIZE  = 32

LABEL_COL   = "gold_label"
TEXT_COL    = "sentence"

LABEL_MAP   = {"entailment": 0, "neutral": 1, "contradiction": 2}
LABEL_NAMES = ["entailment", "neutral", "contradiction"]
COLORS      = ["#2196F3", "#FF9800", "#E91E63"]   # blue, orange, pink

EMBEDDINGS_DIR = REPO_ROOT / "data" / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = EMBEDDINGS_DIR / "snli_ve_clip.npz"

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Repo root : {REPO_ROOT}")
print(f"Output    : {OUTPUT_PATH}")
print(f"Device    : {DeviceManager.resolve('auto')}")

## 1 · Load SNLI-VE and inspect schema

In [ ]:
print(f"Loading {HF_ID} (split={SPLIT}) …")
raw = load_dataset(HF_ID, split=SPLIT)

print(f"\nDataset size  : {len(raw):,} rows")
print(f"Column names  : {raw.column_names}")
print(f"Features      : {raw.features}")

In [ ]:
# Inspect a single example
sample = raw[0]
print(f"Flickr30K_ID : {sample['Flickr30K_ID']}")
print(f"gold_label   : {sample['gold_label']}")
print(f"sentence     : {sample['sentence']}")
print(f"image type   : {type(sample['image']).__name__}")
print(f"image size   : {sample['image'].size}  (W×H)")

sample['image']

In [ ]:
# Label distribution in the full train split
from collections import Counter

label_counts = Counter(raw[LABEL_COL])
print("Label distribution (full train split):")
for lbl, cnt in sorted(label_counts.items()):
    print(f"  {lbl:<15} {cnt:>7,}")

## 2 · Build balanced subset (1 000 per class)

In [ ]:
INVALID_LABELS = {"-", "", "-1"}

def resolve_label(raw_label) -> int | None:
    """Maps string/int label to int, returns None for invalid entries."""
    if isinstance(raw_label, int):
        return raw_label if raw_label >= 0 else None
    s = str(raw_label).strip().lower()
    if s in INVALID_LABELS:
        return None
    return LABEL_MAP.get(s)


def balanced_subsample(dataset, n_per_class: int, seed: int, label_col: str):
    rng = random.Random(seed)

    label_to_indices: dict[int, list[int]] = defaultdict(list)
    skipped = 0
    for idx, raw_lbl in enumerate(dataset[label_col]):
        lbl = resolve_label(raw_lbl)
        if lbl is None:
            skipped += 1
            continue
        label_to_indices[lbl].append(idx)

    if skipped:
        print(f"  Skipped {skipped} rows with invalid labels")

    selected: list[int] = []
    for lbl in sorted(label_to_indices):
        name = LABEL_NAMES[lbl]
        available = len(label_to_indices[lbl])
        chosen = rng.sample(label_to_indices[lbl], n_per_class)
        selected.extend(chosen)
        print(f"  class {lbl} ({name:<15}): {available:>7,} available → {n_per_class} selected")

    selected.sort()
    return dataset.select(selected)


print(f"Building balanced subset: {N_PER_CLASS} per class, seed={SEED}")
subset = balanced_subsample(raw, n_per_class=N_PER_CLASS, seed=SEED, label_col=LABEL_COL)

labels_int = np.array([resolve_label(v) for v in subset[LABEL_COL]], dtype=np.int64)
unique, counts = np.unique(labels_int, return_counts=True)
print(f"\nSubset total : {len(subset):,} rows")
print(f"Distribution : { {LABEL_NAMES[u]: int(c) for u, c in zip(unique, counts)} }")

In [ ]:
# Quick visual check — one image per class
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
shown = {}
for i, row in enumerate(subset):
    lbl = resolve_label(row[LABEL_COL])
    if lbl not in shown:
        shown[lbl] = row
    if len(shown) == 3:
        break

for ax, (lbl, row) in zip(axes, sorted(shown.items())):
    ax.imshow(row["image"])
    ax.set_title(f"class {lbl}: {LABEL_NAMES[lbl]}\n'{row[TEXT_COL][:60]}…'", fontsize=8)
    ax.axis("off")

plt.suptitle("One sample per class", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 3 · Load CLIP

CLIP is used as a **frozen feature extractor** — no fine-tuning required.  
The model was already trained to align images and text in a shared embedding space, which is exactly what we want to probe.

In [ ]:
print(f"Loading {MODEL_NAME} …")
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME)

DEVICE = DeviceManager.resolve("auto")
DeviceManager.prepare_model(model, DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel        : {MODEL_NAME}")
print(f"Parameters   : {n_params:,}")
print(f"Device       : {DEVICE}")
print(f"Mode         : eval (frozen — no fine-tuning)")

## 4 · Extract embeddings

In [ ]:
def coerce_to_pil(img) -> Image.Image:
    if not isinstance(img, Image.Image):
        raise TypeError(f"Expected PIL.Image, got {type(img).__name__}")
    return img.convert("RGB") if img.mode != "RGB" else img


def l2_normalize(x: torch.Tensor) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def extract_embeddings(model, processor, dataset, device, batch_size, label_col, text_col):
    """Returns (image_arr, text_arr, labels_arr) — all float32 numpy, L2-normalised."""
    image_chunks, text_chunks, label_chunks = [], [], []
    n = len(dataset)
    n_batches = (n + batch_size - 1) // batch_size

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_idx = start // batch_size + 1
        rows = dataset[start:end]

        images, texts, labels = [], [], []
        for img_field, text, raw_lbl in zip(rows["image"], rows[text_col], rows[label_col]):
            try:
                images.append(coerce_to_pil(img_field))
            except Exception as e:
                print(f"  [batch {batch_idx}] skipping bad image: {e}")
                continue
            lbl = resolve_label(raw_lbl)
            if lbl is None:
                images.pop()
                continue
            texts.append(text)
            labels.append(lbl)

        if not images:
            continue

        inputs = processor(
            text=texts, images=images,
            return_tensors="pt", padding=True, truncation=True
        )
        pixel_values   = inputs["pixel_values"].to(device)
        input_ids      = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        image_emb = l2_normalize(model.get_image_features(pixel_values=pixel_values))
        text_emb  = l2_normalize(model.get_text_features(
            input_ids=input_ids, attention_mask=attention_mask
        ))

        image_chunks.append(image_emb.detach().cpu().float())
        text_chunks.append(text_emb.detach().cpu().float())
        label_chunks.extend(labels)

        if batch_idx % 10 == 0 or batch_idx == n_batches:
            done = sum(c.shape[0] for c in image_chunks)
            print(f"  batch {batch_idx:>3}/{n_batches} — {done}/{n} samples", flush=True)

    return (
        torch.cat(image_chunks).numpy(),
        torch.cat(text_chunks).numpy(),
        np.array(label_chunks, dtype=np.int64),
    )


print(f"Extracting embeddings ({N_PER_CLASS * 3:,} samples, batch_size={BATCH_SIZE}) …")
with torch.no_grad():
    image_arr, text_arr, labels_arr = extract_embeddings(
        model, processor, subset,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        label_col=LABEL_COL,
        text_col=TEXT_COL,
    )

print(f"\nimage  : {image_arr.shape}  ||·||₂ = {np.linalg.norm(image_arr, axis=1).mean():.4f}")
print(f"text   : {text_arr.shape}   ||·||₂ = {np.linalg.norm(text_arr,  axis=1).mean():.4f}")
print(f"labels : {labels_arr.shape}  classes = {np.unique(labels_arr).tolist()}")

## 5 · Build the four representations

In [ ]:
representations = {
    "image"    : image_arr,
    "text"     : text_arr,
    "concat"   : np.concatenate([image_arr, text_arr], axis=1),
    "pairwise" : np.concatenate(
        [image_arr, text_arr,
         np.abs(image_arr - text_arr),
         image_arr * text_arr],
        axis=1
    ),
}

print("Representation shapes:")
for name, arr in representations.items():
    print(f"  {name:<10}  {str(arr.shape):<15}  dtype={arr.dtype}")

## 6 · Save to disk

In [ ]:
save_features(representations, labels_arr, str(OUTPUT_PATH))

size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f"Saved → {OUTPUT_PATH}  ({size_mb:.1f} MB)")

# Quick sanity check — reload and verify shapes
loaded = np.load(OUTPUT_PATH)
print("\nKeys and shapes in .npz:")
for k in loaded.files:
    print(f"  {k:<30}  {loaded[k].shape}")

## 7 · PCA 2D visualisation

One panel per representation. Each point is one sample, coloured by entailment label.

In [ ]:
def pca_scatter(ax, X: np.ndarray, y: np.ndarray, title: str):
    """Fit PCA on X and plot 2D scatter on ax."""
    pca = PCA(n_components=2, random_state=SEED)
    X2 = pca.fit_transform(X)
    var = pca.explained_variance_ratio_

    for lbl, color, name in zip([0, 1, 2], COLORS, LABEL_NAMES):
        mask = y == lbl
        ax.scatter(
            X2[mask, 0], X2[mask, 1],
            c=color, label=name,
            s=8, alpha=0.55, linewidths=0,
        )

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(f"PC1 ({var[0]:.1%})", fontsize=9)
    ax.set_ylabel(f"PC2 ({var[1]:.1%})", fontsize=9)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.25)
    return var


fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.ravel()

rep_titles = {
    "image"    : "Image (CLIP, 512-d)",
    "text"     : "Text / Hypothesis (CLIP, 512-d)",
    "concat"   : "Concat: image ⊕ text (1 024-d)",
    "pairwise" : "Pairwise: [img, txt, |img−txt|, img⊙txt] (2 048-d)",
}

for ax, (name, title) in zip(axes, rep_titles.items()):
    var = pca_scatter(ax, representations[name], labels_arr, title)
    print(f"{name:<10}  PC1={var[0]:.2%}  PC2={var[1]:.2%}  total={var.sum():.2%}")

# Shared legend
handles = [
    mpatches.Patch(color=c, label=n)
    for c, n in zip(COLORS, LABEL_NAMES)
]
fig.legend(
    handles=handles, title="Label",
    loc="lower center", ncol=3,
    fontsize=10, title_fontsize=10,
    bbox_to_anchor=(0.5, -0.02),
)

fig.suptitle(
    f"SNLI-VE × CLIP — PCA 2D  (N={len(labels_arr):,}, seed={SEED})",
    fontsize=14, fontweight="bold", y=1.01,
)
plt.tight_layout()

plot_path = EMBEDDINGS_DIR / "snli_ve_clip_pca.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"\nPlot saved → {plot_path}")
plt.show()

## 8 · Cosine similarity sanity check

For a well-calibrated CLIP model, **entailment** pairs should have higher image–text cosine similarity than **contradiction** pairs.

In [ ]:
# Per-sample cosine similarity between image and text embeddings
# (both are L2-normalised, so dot product == cosine similarity)
cos_sim = (image_arr * text_arr).sum(axis=1)  # [N]

print("Mean image–text cosine similarity per label:")
for lbl, name in enumerate(LABEL_NAMES):
    mask = labels_arr == lbl
    print(f"  {lbl} {name:<15}  {cos_sim[mask].mean():.4f}  (±{cos_sim[mask].std():.4f})")

# Box plot
fig, ax = plt.subplots(figsize=(7, 4))
data_per_label = [cos_sim[labels_arr == lbl] for lbl in range(3)]
bp = ax.boxplot(data_per_label, patch_artist=True, notch=False)
for patch, color in zip(bp["boxes"], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(LABEL_NAMES, fontsize=10)
ax.set_ylabel("Cosine similarity (image, text)", fontsize=10)
ax.set_title("CLIP image–text alignment by SNLI-VE label", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()